# Video Notebook 1: Qubit & Bloch Sphere — Animated

---

## What This Notebook Does

This notebook generates **animated visualizations** of quantum computing concepts. Each cell produces a `.gif` animation that you can watch inline or save.

### Animations Included

1. **Bloch Sphere Rotation** — Watch a qubit state evolve on the Bloch sphere
2. **Quantum Gate Effects** — H, X, Y, Z, S, T gates animated step-by-step
3. **Superposition Build-up** — From $|0\rangle$ to equal superposition
4. **Measurement Collapse** — Superposition collapsing to $|0\rangle$ or $|1\rangle$
5. **Rabi Oscillation** — Continuous rotation between $|0\rangle$ and $|1\rangle$
6. **Multi-Gate Sequence** — Applying a circuit gate-by-gate on the Bloch sphere

> **Tip**: Run each cell to generate the animation. GIFs are saved to `animations/` folder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, Image, display
import os

# Create output directory
os.makedirs('animations', exist_ok=True)

# =============================================
# BLOCH SPHERE UTILITY FUNCTIONS
# =============================================

def angles_to_bloch(theta, phi):
    """Convert qubit angles (theta, phi) to Bloch sphere (x, y, z)."""
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return x, y, z

def statevector_to_bloch(alpha, beta):
    """Convert statevector |psi> = alpha|0> + beta|1> to Bloch coordinates."""
    # Ensure normalization
    norm = np.sqrt(abs(alpha)**2 + abs(beta)**2)
    alpha, beta = alpha/norm, beta/norm
    
    x = 2 * np.real(np.conj(alpha) * beta)
    y = 2 * np.imag(np.conj(alpha) * beta)
    z = abs(alpha)**2 - abs(beta)**2
    return x, y, z

def draw_bloch_sphere(ax, title='', elev=20, azim=30):
    """Draw a wireframe Bloch sphere on a 3D axis."""
    ax.clear()
    
    # Sphere wireframe
    u = np.linspace(0, 2 * np.pi, 40)
    v = np.linspace(0, np.pi, 20)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color='#cccccc', alpha=0.15, linewidth=0.5)
    
    # Axes
    ax.plot([-1.3, 1.3], [0, 0], [0, 0], 'k-', alpha=0.3, linewidth=0.8)
    ax.plot([0, 0], [-1.3, 1.3], [0, 0], 'k-', alpha=0.3, linewidth=0.8)
    ax.plot([0, 0], [0, 0], [-1.3, 1.3], 'k-', alpha=0.3, linewidth=0.8)
    
    # Equator circle
    theta_eq = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta_eq), np.sin(theta_eq), 0, 'k-', alpha=0.15, linewidth=0.8)
    
    # Labels
    ax.text(0, 0, 1.45, '$|0\\rangle$', fontsize=14, ha='center', fontweight='bold', color='#2196F3')
    ax.text(0, 0, -1.45, '$|1\\rangle$', fontsize=14, ha='center', fontweight='bold', color='#F44336')
    ax.text(1.45, 0, 0, '$|+\\rangle$', fontsize=11, ha='center', color='#4CAF50')
    ax.text(-1.45, 0, 0, '$|-\\rangle$', fontsize=11, ha='center', color='#FF9800')
    ax.text(0, 1.45, 0, '$|+i\\rangle$', fontsize=10, ha='center', color='gray')
    ax.text(0, -1.45, 0, '$|-i\\rangle$', fontsize=10, ha='center', color='gray')
    
    # Styling
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_zlim(-1.5, 1.5)
    ax.set_axis_off()
    ax.view_init(elev=elev, azim=azim)
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

# Gate matrices
I_gate = np.array([[1, 0], [0, 1]])
X_gate = np.array([[0, 1], [1, 0]])
Y_gate = np.array([[0, -1j], [1j, 0]])
Z_gate = np.array([[1, 0], [0, -1]])
H_gate = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
S_gate = np.array([[1, 0], [0, 1j]])
T_gate = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]])

def Rx(theta):
    return np.array([[np.cos(theta/2), -1j*np.sin(theta/2)],
                     [-1j*np.sin(theta/2), np.cos(theta/2)]])

def Ry(theta):
    return np.array([[np.cos(theta/2), -np.sin(theta/2)],
                     [np.sin(theta/2), np.cos(theta/2)]])

def Rz(theta):
    return np.array([[np.exp(-1j*theta/2), 0],
                     [0, np.exp(1j*theta/2)]])

def apply_gate(state, gate):
    return gate @ state

def interpolate_states(state1, state2, n_frames):
    """Smoothly interpolate between two quantum states on the Bloch sphere."""
    bloch1 = np.array(statevector_to_bloch(state1[0], state1[1]))
    bloch2 = np.array(statevector_to_bloch(state2[0], state2[1]))
    
    # Great circle interpolation (SLERP on Bloch sphere)
    frames = []
    for i in range(n_frames):
        t = i / max(n_frames - 1, 1)
        # Smooth easing
        t = 0.5 - 0.5 * np.cos(np.pi * t)
        point = (1 - t) * bloch1 + t * bloch2
        norm = np.linalg.norm(point)
        if norm > 0:
            point = point / norm
        frames.append(point)
    return frames

print('Bloch sphere utilities loaded!')
print('Ready to generate animations.')

## Animation 1: Bloch Sphere State Evolution

Watch the qubit state vector trace a path on the Bloch sphere as it undergoes continuous $R_Y$ rotation:

$$|\psi(t)\rangle = R_Y(\omega t)|0\rangle = \cos\frac{\omega t}{2}|0\rangle + \sin\frac{\omega t}{2}|1\rangle$$

In [ ]:
# Animation 1: Bloch Sphere — Continuous RY Rotation

fig = plt.figure(figsize=(8, 8), facecolor='white')
ax = fig.add_subplot(111, projection='3d')

n_frames = 80
trail_x, trail_y, trail_z = [], [], []

def animate_ry(frame):
    theta = frame * 2 * np.pi / n_frames  # Full rotation
    state = Ry(theta) @ np.array([1, 0], dtype=complex)
    x, y, z = statevector_to_bloch(state[0], state[1])
    
    trail_x.append(x); trail_y.append(y); trail_z.append(z)
    
    draw_bloch_sphere(ax, f'$R_Y$ Rotation: $\\theta = {np.degrees(theta):.0f}\\degree$')
    
    # Draw trail
    if len(trail_x) > 1:
        for i in range(1, len(trail_x)):
            alpha = 0.1 + 0.9 * (i / len(trail_x))
            ax.plot([trail_x[i-1], trail_x[i]], [trail_y[i-1], trail_y[i]], 
                    [trail_z[i-1], trail_z[i]], '-', color='#6c63ff', alpha=alpha, linewidth=2)
    
    # Current state vector
    ax.quiver(0, 0, 0, x, y, z, color='#FF4081', arrow_length_ratio=0.1, linewidth=3)
    ax.scatter([x], [y], [z], c='#FF4081', s=120, edgecolor='black', linewidth=1.5, zorder=10)
    
    # State label
    prob0 = abs(state[0])**2
    prob1 = abs(state[1])**2
    ax.text2D(0.02, 0.95, f'$|\\psi\\rangle = {state[0]:.2f}|0\\rangle + {state[1]:.2f}|1\\rangle$',
              transform=ax.transAxes, fontsize=11, verticalalignment='top',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.text2D(0.02, 0.88, f'P(0)={prob0:.2f}  P(1)={prob1:.2f}',
              transform=ax.transAxes, fontsize=10, verticalalignment='top')

anim1 = animation.FuncAnimation(fig, animate_ry, frames=n_frames, interval=60, blit=False)
anim1.save('animations/01_ry_rotation.gif', writer='pillow', fps=15, dpi=100)
plt.close()
trail_x.clear(); trail_y.clear(); trail_z.clear()

print('Animation saved: animations/01_ry_rotation.gif')
display(Image(filename='animations/01_ry_rotation.gif'))

## Animation 2: Quantum Gate Gallery

Watch what each standard gate does to the $|0\rangle$ state on the Bloch sphere:

| Gate | Matrix | Effect |
|------|--------|--------|
| $X$ | $\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$ | Bit flip: $|0\rangle \to |1\rangle$ |
| $H$ | $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$ | Superposition: $|0\rangle \to |+\rangle$ |
| $Z$ | $\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$ | Phase flip |
| $S$ | $\begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$ | Quarter phase |

In [ ]:
# Animation 2: Gate Effects on |0> — Side by Side

gates = [
    ('X Gate (Bit Flip)', X_gate, '#F44336'),
    ('H Gate (Hadamard)', H_gate, '#4CAF50'),
    ('Z Gate (Phase Flip)', Z_gate, '#2196F3'),
    ('S Gate (Phase)', S_gate, '#FF9800'),
]

fig = plt.figure(figsize=(16, 4.5), facecolor='white')
axes = [fig.add_subplot(1, 4, i+1, projection='3d') for i in range(4)]

state0 = np.array([1, 0], dtype=complex)  # |0>

# Pre-compute all interpolation frames
n_frames_gate = 40
all_gate_frames = []
for name, gate, color in gates:
    final_state = apply_gate(state0, gate)
    frames = interpolate_states(state0, final_state, n_frames_gate)
    all_gate_frames.append(frames)

def animate_gates(frame):
    for idx, (ax, (name, gate, color)) in enumerate(zip(axes, gates)):
        draw_bloch_sphere(ax, name, elev=25, azim=30)
        
        point = all_gate_frames[idx][frame]
        x, y, z = point[0], point[1], point[2]
        
        # Trail
        if frame > 0:
            trail = all_gate_frames[idx][:frame+1]
            tx = [p[0] for p in trail]
            ty = [p[1] for p in trail]
            tz = [p[2] for p in trail]
            ax.plot(tx, ty, tz, '-', color=color, alpha=0.5, linewidth=2)
        
        # State vector
        ax.quiver(0, 0, 0, x, y, z, color=color, arrow_length_ratio=0.12, linewidth=2.5)
        ax.scatter([x], [y], [z], c=color, s=100, edgecolor='black', linewidth=1, zorder=10)

anim2 = animation.FuncAnimation(fig, animate_gates, frames=n_frames_gate, interval=80, blit=False)
anim2.save('animations/02_gate_effects.gif', writer='pillow', fps=12, dpi=100)
plt.close()

print('Animation saved: animations/02_gate_effects.gif')
display(Image(filename='animations/02_gate_effects.gif'))

## Animation 3: Measurement Collapse

A qubit in superposition $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ is measured, collapsing to either $|0\rangle$ or $|1\rangle$ with equal probability.

In [ ]:
# Animation 3: Measurement Collapse

fig = plt.figure(figsize=(14, 6), facecolor='white')
ax_bloch = fig.add_subplot(121, projection='3d')
ax_prob = fig.add_subplot(122)

# Start in |+> state, collapse to |0>
state_plus = H_gate @ np.array([1, 0], dtype=complex)
state_zero = np.array([1, 0], dtype=complex)
state_one = np.array([0, 1], dtype=complex)

# Choose random outcome
np.random.seed(7)
outcome = np.random.choice([0, 1])  # Random measurement outcome
target_state = state_zero if outcome == 0 else state_one

n_frames_meas = 60
# Phase 1: wobble in superposition (frames 0-29)
# Phase 2: collapse (frames 30-59)

collapse_frames = interpolate_states(state_plus, target_state, 30)

def animate_measurement(frame):
    ax_bloch.clear()
    ax_prob.clear()
    
    if frame < 30:
        # Phase 1: Wobbling in superposition
        wobble = 0.05 * np.sin(frame * 0.8)
        x, y, z = statevector_to_bloch(state_plus[0], state_plus[1])
        x += wobble * np.cos(frame * 0.5)
        y += wobble * np.sin(frame * 0.5)
        phase_text = 'Superposition (pre-measurement)'
        p0, p1 = 0.5, 0.5
        vec_color = '#9C27B0'
    else:
        # Phase 2: Collapse
        collapse_idx = frame - 30
        point = collapse_frames[collapse_idx]
        x, y, z = point[0], point[1], point[2]
        phase_text = f'COLLAPSING to |{outcome}>...'
        t = collapse_idx / 29
        if outcome == 0:
            p0 = 0.5 + 0.5 * t
            p1 = 1 - p0
        else:
            p1 = 0.5 + 0.5 * t
            p0 = 1 - p1
        vec_color = '#2196F3' if outcome == 0 else '#F44336'
        if collapse_idx == 29:
            phase_text = f'COLLAPSED to |{outcome}>!'
    
    # Draw Bloch sphere
    draw_bloch_sphere(ax_bloch, phase_text)
    ax_bloch.quiver(0, 0, 0, x, y, z, color=vec_color, arrow_length_ratio=0.12, linewidth=3)
    ax_bloch.scatter([x], [y], [z], c=vec_color, s=120, edgecolor='black', linewidth=1.5, zorder=10)
    
    # Probability bars
    bars = ax_prob.bar(['$|0\\rangle$', '$|1\\rangle$'], [p0, p1], 
                       color=['#2196F3', '#F44336'], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax_prob.set_ylim(0, 1.15)
    ax_prob.set_ylabel('Probability', fontsize=13)
    ax_prob.set_title('Measurement Probabilities', fontsize=13)
    ax_prob.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, [p0, p1]):
        ax_prob.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                    f'{val:.2f}', ha='center', fontsize=14, fontweight='bold')
    
    # Measurement indicator
    if frame >= 28 and frame <= 32:
        ax_prob.text(0.5, 0.5, 'MEASURING', transform=ax_prob.transAxes,
                    fontsize=20, ha='center', va='center', fontweight='bold',
                    color='red', alpha=min(1, (frame-27)*0.3),
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

anim3 = animation.FuncAnimation(fig, animate_measurement, frames=n_frames_meas, interval=100, blit=False)
anim3.save('animations/03_measurement_collapse.gif', writer='pillow', fps=10, dpi=100)
plt.close()

print(f'Animation saved: animations/03_measurement_collapse.gif (collapsed to |{outcome}>)')
display(Image(filename='animations/03_measurement_collapse.gif'))

## Animation 4: Rabi Oscillation

Continuous driving of a qubit between $|0\rangle$ and $|1\rangle$ — the quantum equivalent of a pendulum:

$$|\psi(t)\rangle = \cos(\Omega t/2)|0\rangle - i\sin(\Omega t/2)|1\rangle$$

where $\Omega$ is the Rabi frequency.

In [ ]:
# Animation 4: Rabi Oscillation with Probability Plot

fig = plt.figure(figsize=(15, 6), facecolor='white')
ax_bloch = fig.add_subplot(121, projection='3d')
ax_osc = fig.add_subplot(122)

n_frames_rabi = 100
times = []
prob0_history = []
prob1_history = []
trail_pts = []

def animate_rabi(frame):
    t = frame * 4 * np.pi / n_frames_rabi  # 2 full Rabi cycles
    
    # Rx rotation (Rabi oscillation)
    state = Rx(t) @ np.array([1, 0], dtype=complex)
    x, y, z = statevector_to_bloch(state[0], state[1])
    
    p0 = abs(state[0])**2
    p1 = abs(state[1])**2
    times.append(t)
    prob0_history.append(p0)
    prob1_history.append(p1)
    trail_pts.append((x, y, z))
    
    # Bloch sphere
    draw_bloch_sphere(ax_bloch, f'Rabi Oscillation ($R_X$)', elev=15, azim=30 + frame*0.5)
    
    # Trail
    if len(trail_pts) > 1:
        tx = [p[0] for p in trail_pts]
        ty = [p[1] for p in trail_pts]
        tz = [p[2] for p in trail_pts]
        ax_bloch.plot(tx, ty, tz, '-', color='#6c63ff', alpha=0.4, linewidth=1.5)
    
    ax_bloch.quiver(0, 0, 0, x, y, z, color='#FF4081', arrow_length_ratio=0.1, linewidth=3)
    ax_bloch.scatter([x], [y], [z], c='#FF4081', s=100, edgecolor='black', linewidth=1.5, zorder=10)
    
    # Oscillation plot
    ax_osc.clear()
    ax_osc.plot(times, prob0_history, 'b-', linewidth=2.5, label='$P(|0\\rangle)$')
    ax_osc.plot(times, prob1_history, 'r-', linewidth=2.5, label='$P(|1\\rangle)$')
    ax_osc.scatter([t], [p0], c='blue', s=80, edgecolor='black', zorder=10)
    ax_osc.scatter([t], [p1], c='red', s=80, edgecolor='black', zorder=10)
    ax_osc.set_xlim(0, 4*np.pi)
    ax_osc.set_ylim(-0.05, 1.1)
    ax_osc.set_xlabel('Time ($\\Omega t$)', fontsize=12)
    ax_osc.set_ylabel('Probability', fontsize=12)
    ax_osc.set_title('Rabi Oscillation', fontsize=13)
    ax_osc.legend(fontsize=12, loc='upper right')
    ax_osc.grid(True, alpha=0.3)
    ax_osc.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)

anim4 = animation.FuncAnimation(fig, animate_rabi, frames=n_frames_rabi, interval=60, blit=False)
anim4.save('animations/04_rabi_oscillation.gif', writer='pillow', fps=15, dpi=100)
plt.close()
times.clear(); prob0_history.clear(); prob1_history.clear(); trail_pts.clear()

print('Animation saved: animations/04_rabi_oscillation.gif')
display(Image(filename='animations/04_rabi_oscillation.gif'))

## Animation 5: Multi-Gate Circuit Sequence

Watch a quantum circuit applied gate-by-gate on the Bloch sphere:

$$|0\rangle \xrightarrow{H} |+\rangle \xrightarrow{T} \frac{|0\rangle + e^{i\pi/4}|1\rangle}{\sqrt{2}} \xrightarrow{H} \cos\frac{\pi}{8}|0\rangle + \sin\frac{\pi}{8}|1\rangle \xrightarrow{X} \cdots$$

In [ ]:
# Animation 5: Multi-Gate Circuit on Bloch Sphere

circuit_gates = [
    ('H', H_gate, '#4CAF50'),
    ('T', T_gate, '#FF9800'),
    ('H', H_gate, '#4CAF50'),
    ('X', X_gate, '#F44336'),
    ('S', S_gate, '#2196F3'),
    ('H', H_gate, '#4CAF50'),
]

fig = plt.figure(figsize=(14, 7), facecolor='white')
ax_bloch = fig.add_subplot(121, projection='3d')
ax_circuit = fig.add_subplot(122)

# Pre-compute all states
states_sequence = [np.array([1, 0], dtype=complex)]  # Start at |0>
for _, gate, _ in circuit_gates:
    states_sequence.append(apply_gate(states_sequence[-1], gate))

# Pre-compute interpolations
frames_per_gate = 25
pause_frames = 10
total_frames = len(circuit_gates) * (frames_per_gate + pause_frames)

all_interp = []
for i in range(len(circuit_gates)):
    interp = interpolate_states(states_sequence[i], states_sequence[i+1], frames_per_gate)
    all_interp.append(interp)

full_trail = []

def animate_circuit(frame):
    gate_idx = frame // (frames_per_gate + pause_frames)
    local_frame = frame % (frames_per_gate + pause_frames)
    
    if gate_idx >= len(circuit_gates):
        gate_idx = len(circuit_gates) - 1
        local_frame = frames_per_gate + pause_frames - 1
    
    if local_frame < frames_per_gate:
        point = all_interp[gate_idx][local_frame]
    else:
        point = all_interp[gate_idx][-1]
    
    x, y, z = point[0], point[1], point[2]
    full_trail.append((x, y, z))
    
    gate_name = circuit_gates[gate_idx][0]
    gate_color = circuit_gates[gate_idx][2]
    
    title = f'Applying: {gate_name} (Gate {gate_idx+1}/{len(circuit_gates)})'
    if local_frame >= frames_per_gate:
        title = f'After: {gate_name} (Gate {gate_idx+1}/{len(circuit_gates)})'
    
    draw_bloch_sphere(ax_bloch, title, elev=20, azim=25 + frame * 0.3)
    
    # Full trail
    if len(full_trail) > 1:
        tx = [p[0] for p in full_trail]
        ty = [p[1] for p in full_trail]
        tz = [p[2] for p in full_trail]
        ax_bloch.plot(tx, ty, tz, '-', color='#6c63ff', alpha=0.3, linewidth=1.5)
    
    ax_bloch.quiver(0, 0, 0, x, y, z, color=gate_color, arrow_length_ratio=0.12, linewidth=3)
    ax_bloch.scatter([x], [y], [z], c=gate_color, s=120, edgecolor='black', linewidth=1.5, zorder=10)
    
    # Circuit diagram
    ax_circuit.clear()
    ax_circuit.set_xlim(-0.5, len(circuit_gates) + 0.5)
    ax_circuit.set_ylim(-1, 2)
    ax_circuit.plot([-0.3, len(circuit_gates) + 0.3], [0.5, 0.5], 'k-', linewidth=2)
    ax_circuit.text(-0.4, 0.5, '$|0\\rangle$', fontsize=14, va='center', ha='right', fontweight='bold')
    
    for i, (name, _, color) in enumerate(circuit_gates):
        if i < gate_idx or (i == gate_idx and local_frame >= frames_per_gate):
            # Completed
            box_color = color
            alpha = 0.9
        elif i == gate_idx:
            # Active
            box_color = color
            alpha = 0.5 + 0.4 * np.sin(local_frame * 0.3)  # Pulsing
        else:
            # Future
            box_color = '#cccccc'
            alpha = 0.4
        
        rect = plt.Rectangle((i + 0.15, 0.1), 0.7, 0.8, facecolor=box_color, 
                             edgecolor='black', linewidth=2, alpha=alpha)
        ax_circuit.add_patch(rect)
        ax_circuit.text(i + 0.5, 0.5, name, fontsize=16, ha='center', va='center', 
                       fontweight='bold', color='white' if alpha > 0.6 else 'gray')
    
    ax_circuit.set_title('Circuit Progress', fontsize=13)
    ax_circuit.axis('off')

anim5 = animation.FuncAnimation(fig, animate_circuit, frames=total_frames, interval=70, blit=False)
anim5.save('animations/05_circuit_sequence.gif', writer='pillow', fps=14, dpi=100)
plt.close()
full_trail.clear()

print('Animation saved: animations/05_circuit_sequence.gif')
display(Image(filename='animations/05_circuit_sequence.gif'))

## Animation 6: Quantum State Tomography Visualization

Rotating the Bloch sphere to show all perspectives of a quantum state, with probability bar chart updating in real-time.

In [ ]:
# Animation 6: 360-Degree Bloch Sphere View of Different States

fig = plt.figure(figsize=(10, 8), facecolor='white')
ax = fig.add_subplot(111, projection='3d')

# Multiple states to display simultaneously
display_states = [
    ('$|0\\rangle$', np.array([1, 0], dtype=complex), '#2196F3'),
    ('$|+\\rangle$', H_gate @ np.array([1, 0], dtype=complex), '#4CAF50'),
    ('$|+i\\rangle$', S_gate @ H_gate @ np.array([1, 0], dtype=complex), '#FF9800'),
    ('$T|+\\rangle$', T_gate @ H_gate @ np.array([1, 0], dtype=complex), '#9C27B0'),
]

n_frames_360 = 90

def animate_360(frame):
    azim = frame * 360 / n_frames_360
    draw_bloch_sphere(ax, 'Quantum States Gallery (360\u00b0 View)', elev=20, azim=azim)
    
    for name, state, color in display_states:
        x, y, z = statevector_to_bloch(state[0], state[1])
        ax.quiver(0, 0, 0, x, y, z, color=color, arrow_length_ratio=0.1, linewidth=2.5, alpha=0.8)
        ax.scatter([x], [y], [z], c=color, s=100, edgecolor='black', linewidth=1, zorder=10)
        ax.text(x*1.25, y*1.25, z*1.25, name, fontsize=10, color=color, fontweight='bold')

anim6 = animation.FuncAnimation(fig, animate_360, frames=n_frames_360, interval=50, blit=False)
anim6.save('animations/06_states_360_view.gif', writer='pillow', fps=18, dpi=100)
plt.close()

print('Animation saved: animations/06_states_360_view.gif')
display(Image(filename='animations/06_states_360_view.gif'))

## Summary

### Generated Animations

| File | Description | Frames |
|------|-------------|--------|
| `01_ry_rotation.gif` | Continuous $R_Y$ rotation on Bloch sphere | 80 |
| `02_gate_effects.gif` | X, H, Z, S gate effects side-by-side | 40 |
| `03_measurement_collapse.gif` | Superposition collapsing upon measurement | 60 |
| `04_rabi_oscillation.gif` | Rabi oscillation with probability plot | 100 |
| `05_circuit_sequence.gif` | Multi-gate circuit applied step-by-step | 210 |
| `06_states_360_view.gif` | 360-degree view of multiple quantum states | 90 |

### How to Use These

- **In presentations**: Embed the GIFs directly in slides
- **On GitHub**: GIFs render automatically in README and notebooks
- **In teaching**: Step through concepts visually
- **On social media**: Share to explain quantum concepts

### Customization

Modify parameters in each cell:
- `n_frames` — more frames = smoother but larger file
- `fps` — playback speed
- `dpi` — resolution
- Gate sequences, colors, viewing angles — all customizable